Data Ingestion to Vector DB.

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\harsh\AppData\Local\Temp\ipykernel_2220\1433475898.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_all_pdfs(pdf_directory):
    '''Process all pdf files in a directory. '''
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF in the Directory : 
    pdf_files = list(pdf_dir.glob('**/*.pdf'))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n Processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #adding metadata to pdf files : 
            for doc in documents:
                doc.metadata['source'] = pdf_file.name
                doc.metadata['file_type'] ='pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages ")
        except Exception as e:
            print(f"Error : {e}")
    
    print(f"Total Documents Loaded : {len(all_documents)}")
    return all_documents


In [3]:
all_pdf_documents = process_all_pdfs("../data")

Found 10 PDF files to process

 Processing 00_reading_guide_and_glossary.pdf
Loaded 5 pages 

 Processing 01_attention_and_transformers.pdf
Loaded 5 pages 

 Processing 02_deep_learning_and_language_model_pretraining.pdf
Loaded 5 pages 

 Processing 03_modern_llm_architectures_and_inference.pdf
Loaded 5 pages 

 Processing 04_rag_embeddings_and_ai_applications.pdf
Loaded 6 pages 

 Processing 05_alignment_multimodal_safety_and_frontiers.pdf
Loaded 6 pages 

 Processing ai_basics.pdf
Loaded 1 pages 

 Processing computer_vision.pdf
Loaded 1 pages 

 Processing natural_language_processing.pdf
Loaded 1 pages 

 Processing neural_networks.pdf
Loaded 1 pages 
Total Documents Loaded : 36


In [4]:
#Chunking Using Text Splitter Library.

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Chunking documents into smaller sizes for better RAG performance. """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", '\n', ' ', '']  
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks. ")

    #show sample chunks
    if split_docs:
        print(f"\n Example Chunk : ")
        print(f"Content : {split_docs[0].page_content[:200]}...")
        print(f"Metadata : {split_docs[0].metadata}")
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)

Split 36 documents into 79 chunks. 

 Example Chunk : 
Content : AI Research Series  Educational reference
Page 1
 AI Research Series: Reading Guide and
 Glossary
 A study map from attention mechanisms to production-grade LLM applications
Scope and approach
This d...
Metadata : {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-12T11:03:00+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-10-12T11:03:00+00:00', 'subject': '(unspecified)', 'title': 'AI Research Series: Reading Guide and Glossary', 'trapped': '/False', 'source': '00_reading_guide_and_glossary.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}


Embeddings and VectorDB


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from sentence_transformers import sentence_transformer

class EmbeddingManager:
    def __init__ (self, model_name: str = 'sentence-transformers/all-MiniLM-L6-v2' ):
        '''initialize manager class
        
        Args: 
            Hugging face model minilml6 for sentence embeddings
        '''

        self.model_name = model_name
        self.model = None
        self._load_Model()


    def _load_Model(self):
        """Create a function for loading a Embeddings model : """

        try:
            self.model = SentenceTransformer(self.model_name)  
            print(f"Model Initialized with Embedding Dimensions : {self.model.get_embedding_dimension()}")

        except Exception as e:
            print("Exception Occured" ,e) 
            raise

    def generate_Embeddings(self, texts: List[str]) -> np.ndarray:

        """ Generates a set of embeddings based upon input list of charactrs or texts """

        if not self.model:
            raise ValueError("model not loaded")

        print(f"Generating Embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with Shape {embeddings.shape} of dimensions. ")
        
        return embeddings 

embedding_manager= EmbeddingManager()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1481.62it/s]


Model Initialized with Embedding Dimensions : 384


In [8]:
embedding_manager

In [9]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-12T11:03:00+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-10-12T11:03:00+00:00', 'subject': '(unspecified)', 'title': 'AI Research Series: Reading Guide and Glossary', 'trapped': '/False', 'source': '00_reading_guide_and_glossary.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='AI Research Series \x7f Educational reference\nPage 1\n AI Research Series: Reading Guide and\n Glossary\n A study map from attention mechanisms to production-grade LLM applications\nScope and approach\nThis document is a structured technical learning guide, written to connect foundational ideas to modern practice. It\ndistinguishes established concepts from fast-moving research directions. It is not a systematic literature review and\nshould be read alongside the original papers and documentation in the references.\nHow to use this gu

In [10]:
###Convert the text to embeddings : ###

texts=[doc.page_content for doc in chunks]

In [11]:
texts

['AI Research Series \x7f Educational reference\nPage 1\n AI Research Series: Reading Guide and\n Glossary\n A study map from attention mechanisms to production-grade LLM applications\nScope and approach\nThis document is a structured technical learning guide, written to connect foundational ideas to modern practice. It\ndistinguishes established concepts from fast-moving research directions. It is not a systematic literature review and\nshould be read alongside the original papers and documentation in the references.\nHow to use this guide\nRead sequentially for a course-like progression. For each topic, focus on (1) the problem it solves, (2) the mechanism, (3)\ntrade-offs, and (4) how to test whether it works in a real system.',
 'AI Research Series \x7f Educational reference\nPage 2\nHow the six-document series fits together\n PDF\nMain focus\nRead when you want to understand\n01\nAttention and Transformers\nThe core mechanism behind Transformer models\n02\nDeep learning and pretra

In [12]:
import os

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 20


In [13]:
### Convert the text to embeddings

texts = [doc.page_content for doc in chunks]

### Generate the embeddings : 

embeddings = embedding_manager.generate_Embeddings(texts)

### store embeddings in the Vector DB:

vectorstore.add_documents(chunks, embeddings)

Generating Embeddings for 79 texts...


Batches: 100%|██████████| 3/3 [00:06<00:00,  2.19s/it]

Generated embeddings with Shape (79, 384) of dimensions. 
Adding 79 documents to vector store...
Successfully added 79 documents to vector store
Total documents in collection: 99


In [ ]:
class RagRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    # def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
    #     """
    #     Retrieve relevant documents for a query.
    #     """
    #     print(f"Retrieving documents for query: '{query}'")
    #     print(f"Top K: {top_k}, Score threshold: {score_threshold}")

    #     # Use the actual embedding method defined in EmbeddingManager
    #     query_embedding = self.embedding_manager.generate_Embeddings([query])[0]

    #     try:
    #         results = self.vector_store.collection.query(
    #             query_embeddings=[query_embedding.tolist()],
    #             n_results=top_k
    #         )

    #         retrieved_docs = []

    #         if results.get("documents") and results["documents"]:
    #             documents = results["documents"][0]
    #             metadatas = results["metadatas"][0]
    #             distances = results["distances"][0]
    #             ids = results["ids"][0]

    #             for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
    #                 similarity_score = 1 - distance

    #                 if similarity_score >= score_threshold:
    #                     retrieved_docs.append({
    #                         "id": doc_id,
    #                         "content": document,
    #                         "metadata": metadata,
    #                         "similarity_score": similarity_score,
    #                         "distance": distance,
    #                         "rank": i + 1
    #                     })

    #             print(f"Retrieved {len(retrieved_docs)} documents after filtering")
    #             return retrieved_docs

    #         print("No documents found")
    #         return []
    
        def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0):
            print(f"Query: {query!r}")

            print("Collection document count:", self.vector_store.collection.count())

            query_embedding = self.embedding_manager.generate_Embeddings([query])[0]

            print("Query embedding shape:", query_embedding.shape)

            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            print("Raw IDs:", results.get("ids"))
            print("Raw distances:", results.get("distances"))
            print("Raw documents:", results.get("documents"))

            retrieved_docs = []

            documents = (results.get("documents") or [[]])[0]
            metadatas = (results.get("metadatas") or [[]])[0]
            distances = (results.get("distances") or [[]])[0]
            ids = (results.get("ids") or [[]])[0]

            for doc_id, document, metadata, distance in zip(
                ids, documents, metadatas, distances
            ):
                # Diagnostic only: do not filter by score yet.
                retrieved_docs.append({
                    "id": doc_id,
                    "content": document,
                    "metadata": metadata or {},
                    "distance": float(distance),
                    "rank": len(retrieved_docs) + 1
                })

            print("Documents returned:", len(retrieved_docs))
            return retrieved_docs


        # except Exception as e:
        #     print(f"Error during retrieval: {e}")
        #     return []

In [15]:
rag_retriever = RagRetriever(vectorstore, embedding_manager)

#Integration Vectordb Context pipeline with LLM Output#

In [16]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name = "openai/gpt-oss-20b",temperature=0.1, max_tokens = 1024)

# 2. Simple RAG Function : 

def rag_simple(query,retriver, llm, top_k = 3 ):
    ##retrieve the context
    results = retriver.retrieve(query, top_k = top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevent context found"
    # "generate the answer using Groq LLM "
    prompt = """Use the following contxt to answer the question concisely.
                context : {context}
                question : {query}
                answer : """


    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content

In [17]:
answer = rag_simple("What is machine learning ?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is machine learning ?'
Top K: 3, Score threshold: 0.0
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.68it/s]

Generated embeddings with Shape (1, 384) of dimensions. 
Retrieved 3 documents after filtering


Machine learning is a branch of artificial intelligence in which algorithms learn patterns and make predictions or decisions from data, rather than being explicitly programmed for each task. In computer‑vision, for example, deep‑learning models are trained on large sets of labeled images to recognize edges, shapes, textures, and object parts.


Enhanced RAG Pipeline Features

In [ ]:
def rag_advanced(query, retriever, llm, top_k = 5, min_score = 0.2, return_context = False):
    """
    RAG Pipeline with extra features:
    - Returns answer, Sources, confidence score, and optionally full context,
    """
    results = retriever.retrieve(query, top_k = top_k, score_threshold = min_score)
    if not results:
        return {'answer':'No Relevent context Found', 'sources': [], "confidence": 0.0, "context": ''}

    #Prepare  context and source : 

    context = "\n\n".join([doc['content'] for doc in results])
    sources = sources = [
    {
        'source': doc['metadata'].get(
            'source_file',
            doc['metadata'].get('source', 'unknown')
        ),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] + '...'
    }
    for doc in results
]
    confidence = max([doc['similarity_score'] for doc in results])

    #Generate answer
    prompt = f"""Use the Following Context to answer the question concisely, \n {context}, \n\nQuestion : {query} \n\nAnswer:"""
    response = llm.invoke(
    prompt.format(context=context, query=query)
)

    output = {
        'answer' : response.content,
        'sources' : sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

In [19]:
#
result = rag_advanced("What is machine learning ?", rag_retriever, llm, top_k=3, min_score = 0.1, return_context = True)

print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview", result['context'][:3000])

Retrieving documents for query: 'What is machine learning ?'
Top K: 3, Score threshold: 0.1
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.42it/s]

Generated embeddings with Shape (1, 384) of dimensions. 
Retrieved 0 documents after filtering
Answer: No Relevent context Found
Sources: []
Confidence: 0.0
Context Preview 


In [20]:

def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0):
    print(f"Query: {query!r}")

    print("Collection document count:", self.vector_store.collection.count())

    query_embedding = self.embedding_manager.generate_Embeddings([query])[0]

    print("Query embedding shape:", query_embedding.shape)

    results = self.vector_store.collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    print("Raw IDs:", results.get("ids"))
    print("Raw distances:", results.get("distances"))
    print("Raw documents:", results.get("documents"))

    retrieved_docs = []

    documents = (results.get("documents") or [[]])[0]
    metadatas = (results.get("metadatas") or [[]])[0]
    distances = (results.get("distances") or [[]])[0]
    ids = (results.get("ids") or [[]])[0]

    for doc_id, document, metadata, distance in zip(
        ids, documents, metadatas, distances
    ):
        # Diagnostic only: do not filter by score yet.
        retrieved_docs.append({
            "id": doc_id,
            "content": document,
            "metadata": metadata or {},
            "distance": float(distance),
            "rank": len(retrieved_docs) + 1
        })

    print("Documents returned:", len(retrieved_docs))
    return retrieved_docs
